<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/Capstone%20Projects/Personal%20Expense%20Monitoring%20System%20/Week%203/pyspark_anomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Week 3 - PySpark**

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

spark = (SparkSession.builder.appName("expense-anomaly")
         .master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate())
print("Spark", spark.version)

Spark 4.0.4


In [2]:
N_USERS, TXN, MONTHS = 2000, 40, 6
START = F.lit("2026-04-01").cast("date")
cats = ["Groceries","Utilities","Transport","Dining","Entertainment","Health","Shopping","Education"]
cat_arr = F.array(*[F.lit(c) for c in cats])

users = (spark.range(1, N_USERS + 1)
         .select(F.col("id").cast("int").alias("user_id"),
                 (F.rand(1) * 6000 + 3000).cast("int").alias("monthly_income")))
users = users.withColumn("monthly_budget", F.round(F.col("monthly_income") * 0.7, 0))

txn = (spark.range(N_USERS * MONTHS * TXN)
       .select((F.col("id") % N_USERS + 1).cast("int").alias("user_id"),
               F.element_at(cat_arr, (F.rand(2) * len(cats)).cast("int") + 1).alias("category"),
               F.round(F.abs(F.randn(3) * 30 + 50) + 3, 2).alias("amount"),
               F.date_add(START, (F.rand(4) * 180).cast("int")).alias("date")))

rent = (users.crossJoin(spark.range(MONTHS).withColumnRenamed("id", "m"))
        .select("user_id", F.lit("Rent").alias("category"),
                F.round(F.col("monthly_income") * 0.3, 2).alias("amount"),
                F.add_months(START, F.col("m").cast("int")).alias("date")))

spikes = (users.filter(F.col("user_id") % 97 == 0)
          .select("user_id", F.lit("Shopping").alias("category"),
                  F.round(F.col("monthly_income") * 0.8, 2).alias("amount"),
                  F.date_add(START, (F.rand(7) * 150).cast("int")).alias("date")))

df = (txn.unionByName(rent).unionByName(spikes)
      .withColumn("expense_id", F.monotonically_increasing_id()))
print("rows:", df.count())

rows: 492020


In [4]:
df.printSchema()
df.show(5)
print("users:", df.select("user_id").distinct().count())

root
 |-- user_id: integer (nullable = true)
 |-- category: string (nullable = false)
 |-- amount: double (nullable = true)
 |-- date: date (nullable = true)
 |-- expense_id: long (nullable = false)

+-------+-------------+------+----------+----------+
|user_id|     category|amount|      date|expense_id|
+-------+-------------+------+----------+----------+
|      1|Entertainment| 17.44|2026-09-19|         0|
|      2|    Transport|  77.9|2026-04-12|         1|
|      3|       Dining| 56.46|2026-04-27|         2|
|      4|       Dining|  54.8|2026-06-16|         3|
|      5|    Education|  44.3|2026-04-02|         4|
+-------+-------------+------+----------+----------+
only showing top 5 rows
users: 2000


In [7]:
df = df.withColumn("month", F.date_format("date", "yyyy-MM"))

monthly = (df.groupBy("user_id", "month")
             .agg(F.round(F.sum("amount"), 2).alias("total_spend"),
                  F.count("*").alias("txn_count")).cache())
monthly.orderBy("user_id", "month").show(10)

+-------+-------+-----------+---------+
|user_id|  month|total_spend|txn_count|
+-------+-------+-----------+---------+
|      1|2026-04|    4147.99|       40|
|      1|2026-05|     3970.2|       38|
|      1|2026-06|    5168.18|       54|
|      1|2026-07|    4497.42|       42|
|      1|2026-08|    3745.97|       33|
|      1|2026-09|    3851.49|       39|
|      2|2026-04|     3732.9|       36|
|      2|2026-05|    3829.38|       40|
|      2|2026-06|    4454.74|       50|
|      2|2026-07|    3591.38|       37|
+-------+-------+-----------+---------+
only showing top 10 rows


In [11]:
med = monthly.groupBy("user_id").agg(F.percentile_approx("total_spend", 0.5).alias("median_spend"))
monthly_flags = (monthly.join(med, "user_id")
                 .withColumn("spike_ratio", F.round(F.col("total_spend") / F.col("median_spend"), 2))
                 .withColumn("monthly_spike", F.col("spike_ratio") > 1.5))
monthly_flags.filter("monthly_spike").orderBy(F.desc("spike_ratio")).show(10)


Z_THRESHOLD = 5
non_rent = df.filter(F.col("category") != "Rent")
ustats = non_rent.groupBy("user_id").agg(F.avg("amount").alias("mu"), F.stddev("amount").alias("sigma"))
large = (non_rent.join(ustats, "user_id")
         .withColumn("z_score", F.round((F.col("amount") - F.col("mu")) / F.col("sigma"), 2))
         .filter(F.col("z_score") > Z_THRESHOLD)
         .select("user_id", "expense_id", "category", "amount", "date", "z_score"))
large.orderBy(F.desc("z_score")).show(10)

+-------+-------+-----------+---------+------------+-----------+-------------+
|user_id|  month|total_spend|txn_count|median_spend|spike_ratio|monthly_spike|
+-------+-------+-----------+---------+------------+-----------+-------------+
|    679|2026-05|   12357.43|       45|     4563.39|       2.71|         true|
|   1455|2026-07|    9787.06|       48|     4019.94|       2.43|         true|
|   1552|2026-06|    11532.9|       34|      4888.5|       2.36|         true|
|   1940|2026-06|   10801.45|       41|     4584.41|       2.36|         true|
|    776|2026-08|    9648.49|       43|     4106.99|       2.35|         true|
|    970|2026-07|    8992.37|       43|     3820.19|       2.35|         true|
|   1164|2026-07|   10627.57|       45|     4535.73|       2.34|         true|
|   1358|2026-06|    7473.37|       45|     3296.69|       2.27|         true|
|   1649|2026-07|    7592.67|       48|     3353.75|       2.26|         true|
|   1746|2026-04|    9397.04|       39|     4190.08|

In [12]:
spike_users = (monthly_flags.filter("monthly_spike")
               .groupBy("user_id").agg(F.count("*").alias("spike_months"),
                                       F.max("spike_ratio").alias("max_spike_ratio")))
large_users = (large.groupBy("user_id")
               .agg(F.count("*").alias("large_txns"), F.max("amount").alias("largest_txn")))

unusual = (spike_users.join(large_users, "user_id", "full_outer").fillna(0)
           .join(users.select("user_id", "monthly_income"), "user_id", "left")
           .orderBy(F.desc("largest_txn")))
unusual.show(25, truncate=False)
print("Users flagged:", unusual.count(), "of", users.count())

+-------+------------+---------------+----------+-----------+--------------+
|user_id|spike_months|max_spike_ratio|large_txns|largest_txn|monthly_income|
+-------+------------+---------------+----------+-----------+--------------+
|1552   |1           |2.36           |1         |7162.4     |8953          |
|679    |1           |2.71           |1         |7084.8     |8856          |
|1940   |1           |2.36           |1         |6478.4     |8098          |
|1164   |1           |2.34           |1         |5926.4     |7408          |
|776    |1           |2.35           |1         |5416.8     |6771          |
|1746   |1           |2.24           |1         |5316.8     |6646          |
|1455   |1           |2.43           |1         |5184.0     |6480          |
|970    |1           |2.35           |1         |4815.2     |6019          |
|291    |1           |1.98           |1         |4407.2     |5509          |
|1067   |1           |2.17           |1         |4308.0     |5385          |

In [13]:
unusual.toPandas().to_csv("unusual_spending_users.csv", index=False)
large.toPandas().to_csv("large_transactions.csv", index=False)
monthly.toPandas().to_csv("monthly_spend_per_user.csv", index=False)

from google.colab import files
for f in ["unusual_spending_users.csv", "large_transactions.csv", "monthly_spend_per_user.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>